# CV4IS Übungsprojekt — Part 2
## Transfer Learning and Robustness under Corruptions

In this second component, you fine-tune a pretrained model for the same binary task as in Part 1:

```text
0 = good
1 = defective
```

You then evaluate the model on corrupted images and compare the results with your CNN from scratch.

This notebook assumes that you already solved the basic dataset, dataloader, training, and evaluation steps in Part 1.

## What is reused from Part 1?

Reuse your own implementation where appropriate:

- imports,
- dataset loading,
- dataloaders,
- training loop,
- prediction/evaluation functions,
- plots and metrics,
- `split.csv`.

Do **not** create a new split here. Use the same `split.csv` from Part 1 so that your scratch CNN and fine-tuned model are evaluated on the same images.

## Expected data

Clean data:

```text
data/hazelnut_binary_raw/
  good/
  defective/
  metadata.csv
```

Corrupted data:

```text
data/hazelnut_binary_corruptions/
  defocus_blur/
    severity_0.50/
    severity_0.75/
  gaussian_noise/
    severity_0.50/
    severity_0.75/
  metadata.csv
```

## 0. Imports

Import what you need. You can reuse the imports from Part 1 and add the required `torchvision` imports for pretrained models.

In [ ]:
# TODO: import the libraries you need

## 1. Configuration

Set the paths and hyperparameters for transfer learning.

In [ ]:
CLEAN_DATA_ROOT = Path("data/hazelnut_binary_raw")
CORRUPTED_DATA_ROOT = Path("data/hazelnut_binary_corruptions")
SPLIT_CSV = Path("split.csv")

SEED = 42
IMAGE_SIZE = 224

# TODO: choose suitable values
BATCH_SIZE = None
LEARNING_RATE = None
NUM_EPOCHS = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# TODO: set random seeds as in Part 1

## 2. Load the split from Part 1

Load `split.csv` and reuse it for all experiments in this notebook.

In [ ]:
# TODO: load split.csv

split_df = None

# TODO: inspect the split and verify that train/val/test are present

## 3. Dataset and dataloaders

Reuse or adapt your dataset and dataloader implementation from Part 1.

For pretrained ImageNet models, use ImageNet-style normalization and resize the images to the input size expected by your model.

In [ ]:
# TODO: define or reuse your dataset class

# TODO: define train and evaluation transforms
# Hint: pretrained ImageNet models usually require ImageNet normalization.

train_transform = None
eval_transform = None

# TODO: create train/val/test datasets and dataloaders

train_loader = None
val_loader = None
test_loader = None

## 4. Build a pretrained model

Replace the final classification layer so that the model predicts two classes.

You may use the helper function below or adapt it for a different pretrained model.

In [ ]:
def build_resnet18_binary(freeze_backbone: bool = True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 2)

    return model


model = build_resnet18_binary(freeze_backbone=True).to(DEVICE)

# TODO: inspect the model and count trainable parameters

## 5. Loss function and optimizer

Define the loss function and optimizer.

Consider whether class imbalance should be handled here.

In [ ]:
# TODO: define criterion and optimizer

criterion = None
optimizer = None

## 6. Fine-tune the model

Reuse or adapt your training and evaluation functions from Part 1.

In [ ]:
# TODO: reuse or implement:
# - train_one_epoch(...)
# - evaluate(...)
# - predict(...)

# TODO: train the pretrained model

history = None

## 7. Evaluate on the clean test set

Evaluate the fine-tuned model on the same test split used for your scratch CNN.

Report at least precision, recall, F1-score, and the confusion matrix.

In [ ]:
# TODO: evaluate on the clean test set

# TODO: print metrics and plot the confusion matrix

## 8. Save the fine-tuned model

Save the model checkpoint if you want to submit or compare it later.

In [ ]:
# TODO: save your fine-tuned model checkpoint
# Recommended filename:
# finetuned_model.pt

## 9. Corrupted dataset class

The corrupted dataset has the same relative image paths as the clean dataset, but nested under:

```text
<corruption_name>/<severity>/...
```

The class below handles this folder structure.

In [ ]:
class CorruptedHazelnutDataset(Dataset):
    def __init__(
        self,
        corrupted_root,
        clean_split_df,
        split_name,
        corruption,
        severity,
        transform=None,
    ):
        self.corrupted_root = Path(corrupted_root)
        self.clean_split_df = clean_split_df[
            clean_split_df["split"] == split_name
        ].reset_index(drop=True)
        self.corruption = corruption
        self.severity_dir = f"severity_{severity:.2f}"
        self.transform = transform

    def __len__(self):
        return len(self.clean_split_df)

    def __getitem__(self, idx):
        row = self.clean_split_df.iloc[idx]

        relative_path = Path(row["relative_path"])
        image_path = (
            self.corrupted_root
            / self.corruption
            / self.severity_dir
            / relative_path
        )

        image = Image.open(image_path).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

## 10. Robustness evaluation

Evaluate the fine-tuned model on the corrupted test images.

Use the same test split as before.

In [ ]:
corruptions = ["defocus_blur", "gaussian_noise"]
severities = [0.50, 0.75]

robustness_rows = []

for corruption in corruptions:
    for severity in severities:
        corrupted_dataset = CorruptedHazelnutDataset(
            corrupted_root=CORRUPTED_DATA_ROOT,
            clean_split_df=split_df,
            split_name="test",
            corruption=corruption,
            severity=severity,
            transform=eval_transform,
        )

        corrupted_loader = DataLoader(
            corrupted_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )

        # TODO: evaluate the model on corrupted_loader
        # TODO: append the metrics to robustness_rows

robustness_df = None

# TODO: display and save robustness_df

## 11. Visualize robustness results

Plot how the metrics change with corruption severity.

In [ ]:
# TODO: plot robustness metrics

## 12. Compare with your scratch CNN

Compare the fine-tuned model with your CNN from scratch.

Discuss:

- clean test performance,
- corrupted test performance,
- defective recall,
- which corruption is most harmful,
- whether transfer learning improves robustness.